# 🎯 EventHub Custom AI - Llama 3 Training on Colab

This notebook trains a custom Llama 3 model fine-tuned for EventHub using Google Colab's free GPU.

**⚡ IMPORTANT: Enable GPU Runtime**
1. Click `Runtime` → `Change runtime type`
2. Select `T4 GPU` (free tier)
3. Click `Save`

**⏱️ Training Time:** 2-3 hours on T4 GPU

**📋 Requirements:**
- Google account
- Hugging Face account with Llama 3 access
- HF token from: https://huggingface.co/settings/tokens

## 📦 Step 1: Check GPU and Install Dependencies

This will verify GPU availability and install all required packages.

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ CUDA version: {torch.version.cuda}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ No GPU found! Please enable GPU in Runtime settings.")

In [ ]:
# Install dependencies
print("📦 Installing dependencies... (this may take 5-10 minutes)\n")

# Use Colab's PyTorch (already optimized for CUDA 12.x)
# Install compatible versions for CUDA 12.x
!pip install -q transformers==4.46.0
!pip install -q accelerate==1.0.1
!pip install -q peft==0.13.0
!pip install -q bitsandbytes==0.44.0  # Supports CUDA 12.x
!pip install -q datasets==3.0.1
!pip install -q trl==0.11.1
!pip install -q scikit-learn  # For train_test_split

print("\n✅ All dependencies installed!")
print("\n🔍 Verifying bitsandbytes installation...")
!python -m bitsandbytes

## 🔑 Step 2: Hugging Face Authentication

You need to:
1. Get Llama 3 access: https://huggingface.co/meta-llama/Meta-Llama-3-8B
2. Get your token: https://huggingface.co/settings/tokens
3. Enter it below when prompted

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import getpass

print("🔑 Hugging Face Login\n")
print("Get your token from: https://huggingface.co/settings/tokens\n")

# Try to get token from Colab secrets first
try:
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Using token from Colab secrets")
except:
    # Otherwise, ask for it
    hf_token = getpass.getpass("Enter your Hugging Face token: ")

login(token=hf_token)
print("\n✅ Successfully logged in to Hugging Face!")

## 📊 Step 3: Prepare Training Data

Generate EventHub-specific training examples in Llama 3 format.

In [ ]:
import json
import os
from typing import List, Dict

class EventHubDataPrep:
    """Prepare training data for EventHub custom AI."""

    def __init__(self):
        self.system_prompt = """You are an intelligent assistant for EventHub, a premier event management platform in Nigeria. Help users discover events, book tickets, and answer questions about the platform. Be friendly, helpful, and enthusiastic about events!"""

    def create_instruction_dataset(self) -> List[Dict]:
        """Create instruction-following dataset from EventHub data."""
        dataset = []
        dataset.extend(self.event_discovery_examples())
        dataset.extend(self.event_details_examples())
        dataset.extend(self.booking_assistance_examples())
        dataset.extend(self.support_examples())
        dataset.extend(self.general_knowledge_examples())
        return dataset

    def event_discovery_examples(self) -> List[Dict]:
        return [
            {
                "instruction": "Show me upcoming tech events in Lagos",
                "output": "I found several exciting tech events in Lagos:\n\n1. **Tech Conference 2025** (Dec 15, 2025)\n   Location: Lagos Convention Center\n   Price: ₦15,000\n   Featuring industry leaders discussing AI, Web3, and emerging technologies.\n\n2. **StartUp Pitch Night** (Dec 20, 2025)\n   Location: Innovation Center\n   Price: ₦5,000\n   Watch innovative startups pitch to investors!\n\nWould you like more details or help booking tickets?"
            },
            {
                "instruction": "Find music concerts this month",
                "output": "Here are amazing music events happening this month:\n\n1. **Afrobeats Festival** (Dec 22, 2025)\n   Venue: Eko Atlantic\n   Tickets: ₦20,000 - ₦50,000\n   Top Nigerian artists performing!\n\n2. **Jazz Night Lagos** (Dec 28, 2025)\n   Venue: Terra Kulture\n   Tickets: ₦8,000\n   Intimate jazz experience with live band.\n\nAll tickets available on EventHub!"
            }
        ]

    def event_details_examples(self) -> List[Dict]:
        return [
            {
                "instruction": "What time does the Tech Conference start?",
                "output": "The Tech Conference 2025 starts at 9:00 AM on December 15, 2025. Registration opens at 8:00 AM. The event runs until 6:00 PM with keynotes, panel discussions, and networking sessions throughout the day."
            },
            {
                "instruction": "Are there any free events?",
                "output": "Yes! Here are some free events on EventHub:\n\n1. **Community Art Exhibition** - Open gallery viewing\n2. **Developer Meetup** - Free for all developers (register required)\n3. **Fitness in the Park** - Weekly community workout\n\nYou still need to register for free events to secure your spot!"
            }
        ]

    def booking_assistance_examples(self) -> List[Dict]:
        return [
            {
                "instruction": "How do I book tickets for an event?",
                "output": "Booking on EventHub is easy! Here's how:\n\n**Step 1:** Browse events or search by category\n**Step 2:** Select your event and click 'Get Tickets'\n**Step 3:** Choose ticket type (Regular/VIP) and quantity\n**Step 4:** Enter your details and payment info\n**Step 5:** Complete payment - you'll get instant confirmation!\n\nWe accept cards, bank transfers, and mobile money. Questions? I'm here to help!"
            },
            {
                "instruction": "Can I get a refund if I can't attend?",
                "output": "Our refund policy:\n\n- **7+ days before event:** Full refund minus ₦500 service fee\n- **3-7 days before:** 50% refund\n- **Less than 3 days:** No refund, but you can transfer to someone else\n\nTo request a refund, go to 'My Tickets' and click 'Request Refund'. Process takes 5-7 business days."
            }
        ]

    def support_examples(self) -> List[Dict]:
        return [
            {
                "instruction": "I haven't received my ticket confirmation email",
                "output": "Let me help! Here's what to do:\n\n1. Check your spam/junk folder\n2. Search your email for 'EventHub' or 'ticket'\n3. Log into your EventHub account and check 'My Tickets'\n\nStill missing? I can help resend it! What's the email you used for booking? You can also contact support@eventhub.ng with your order number."
            },
            {
                "instruction": "How do I contact event organizers?",
                "output": "To contact organizers:\n\n1. Go to the event page\n2. Scroll to 'Organizer Info' section\n3. Click 'Contact Organizer'\n4. Send your message directly\n\nOrganizers typically respond within 24 hours. For booking issues, contact EventHub support instead at support@eventhub.ng"
            }
        ]

    def general_knowledge_examples(self) -> List[Dict]:
        return [
            {
                "instruction": "Tell me about EventHub",
                "output": "EventHub is Nigeria's premier event management platform! 🎉\n\n**For Attendees:**\n- Discover thousands of events\n- Easy online booking\n- Secure payments\n- Digital tickets with QR codes\n\n**For Organizers:**\n- Create and manage events\n- Sell tickets online\n- Real-time analytics\n- Secure payouts\n\n**Our Mission:** Making events accessible to everyone in Nigeria!\n\nWhat would you like to know more about?"
            },
            {
                "instruction": "What types of events are on EventHub?",
                "output": "We have events for everyone!\n\n🎵 **Music:** Concerts, festivals, club nights\n🎨 **Arts:** Exhibitions, theater, comedy shows\n🏃 **Sports:** Marathons, tournaments, fitness classes\n🍽️ **Food:** Food festivals, restaurant openings\n💼 **Business:** Conferences, networking, workshops\n🎓 **Education:** Seminars, training, workshops\n👨‍👩‍👧 **Family:** Kids events, family fun days\n\nBrowse by category or search for what you love!"
            }
        ]

    def format_for_training(self, dataset: List[Dict]) -> List[Dict]:
        """Format dataset for Llama instruction fine-tuning."""
        formatted = []
        for example in dataset:
            text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{self.system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>"""
            formatted.append({"text": text})
        return formatted

# Generate training data
print("📊 Generating EventHub training data...\n")
prep = EventHubDataPrep()
dataset = prep.create_instruction_dataset()
formatted_dataset = prep.format_for_training(dataset)

# Split into train/val
from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(formatted_dataset, test_size=0.2, random_state=42)

# Save to files
os.makedirs('data', exist_ok=True)

with open('data/train.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open('data/val.jsonl', 'w') as f:
    for item in val_data:
        f.write(json.dumps(item) + '\n')

print(f"✅ Created {len(train_data)} training examples")
print(f"✅ Created {len(val_data)} validation examples")
print("\n📝 Sample training example:")
print("-" * 80)
print(train_data[0]['text'][:300] + "...")
print("-" * 80)

## 🎯 Step 4: Load Model and Configure LoRA

Load Llama 3 8B with 4-bit quantization and configure LoRA for efficient fine-tuning.

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch

print("🔄 Loading Llama 3 model... (this takes 5-10 minutes)\n")

model_name = "meta-llama/Meta-Llama-3-8B"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1

print("✅ Model loaded successfully!\n")

# Configure LoRA
print("🔧 Configuring LoRA...\n")

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("✅ LoRA configured!")
print(f"   Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Total params: {total_params:,}")

## 🚀 Step 5: Train the Model

This is the main training step. It will take approximately **2-3 hours** on a T4 GPU.

You can monitor progress in real-time as it trains!

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer

print("📂 Loading training data...\n")

# Load datasets
dataset = load_dataset(
    'json',
    data_files={
        'train': 'data/train.jsonl',
        'validation': 'data/val.jsonl'
    }
)

print(f"✅ Training examples: {len(dataset['train'])}")
print(f"✅ Validation examples: {len(dataset['validation'])}\n")

# Configure training
training_args = TrainingArguments(
    output_dir="./eventhub-llama3",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    logging_steps=10,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    seed=42,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=2048,
    dataset_text_field="text",
    packing=False,
)

print("\n" + "="*80)
print("🚀 Starting training... (2-3 hours on T4 GPU)")
print("="*80 + "\n")

# Train!
trainer.train()

print("\n" + "="*80)
print("✅ Training completed!")
print("="*80)

## 💾 Step 6: Save the Model

Save your fine-tuned model so you can download it.

In [ ]:
print("💾 Saving model...\n")

output_dir = "./eventhub-llama3-final"

# Save model
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")
print("\n📦 Model files:")
!ls -lh {output_dir}

# Zip for download
print("\n📦 Creating zip file for download...")
!zip -r eventhub-llama3-model.zip {output_dir}
print("✅ Created: eventhub-llama3-model.zip")
print("\n💡 Download this file to use with your inference server!")

## 🧪 Step 7: Test the Model

Let's test your fine-tuned model with some EventHub queries!

In [ ]:
from transformers import pipeline

print("🧪 Testing the fine-tuned model...\n")

# Create inference pipeline
text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def test_query(question: str):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an intelligent assistant for EventHub, a premier event management platform in Nigeria. Help users discover events, book tickets, and answer questions about the platform. Be friendly, helpful, and enthusiastic about events!<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"""
    
    result = text_gen(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )
    
    response = result[0]['generated_text'].split('<|start_header_id|>assistant<|end_header_id|>\n\n')[1].split('<|eot_id|>')[0]
    return response

# Test queries
test_questions = [
    "Show me tech events in Lagos",
    "How do I book tickets?",
    "What is EventHub?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}: {question}")
    print('='*80)
    response = test_query(question)
    print(response)

print("\n" + "="*80)
print("✅ Model testing complete!")
print("="*80)

## 📤 Step 8: Download Your Model

Two options to get your model:

### Option 1: Download from Colab (Recommended)

1. Look in the Files panel (📁) on the left
2. Find `eventhub-llama3-model.zip`
3. Right-click → Download

### Option 2: Mount Google Drive and Save

Run the cell below to save to your Google Drive:

In [ ]:
from google.colab import drive

# Mount Google Drive
print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

# Copy model to Drive
print("\n📤 Copying model to Google Drive...")
!cp eventhub-llama3-model.zip /content/drive/MyDrive/

print("\n✅ Model saved to Google Drive!")
print("📁 Location: My Drive/eventhub-llama3-model.zip")
print("\n💡 You can now download it from your Google Drive on any device!")

## 🎉 Next Steps

### After downloading your model:

1. **Extract the zip file** on your local machine:
   ```bash
   unzip eventhub-llama3-model.zip
   ```

2. **Move to EventHub project:**
   ```bash
   mv eventhub-llama3-final ~/eventmanagement/ml/models/eventhub-llama3
   ```

3. **Start inference server:**
   ```bash
   cd ~/eventmanagement
   python3 ml/inference_server.py
   ```

4. **Update .env.local:**
   ```bash
   echo "CUSTOM_AI_URL=http://localhost:8000" > .env.local
   ```

5. **Start EventHub:**
   ```bash
   npm run dev
   ```

6. **Look for the green sparkles icon (⚡)** in the app!

---

## 📊 Training Summary

- **Model:** Llama 3 8B
- **Method:** LoRA fine-tuning with 4-bit quantization
- **Data:** EventHub-specific instruction-following examples
- **Epochs:** 3
- **Training Time:** ~2-3 hours on T4 GPU
- **Model Size:** ~3-4 GB (compressed)

## 🎯 What You Built

✅ Custom Llama 3 model fine-tuned for EventHub

✅ Specialized in event discovery, booking, and support

✅ Free inference (no API costs!)

✅ Runs on your own hardware

✅ Fully customizable

---

**Congratulations! You've successfully trained your EventHub AI! 🎉**